In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
import os
import colorcet as cc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import harmonypy as hm
import scrublet as scr
import traceback
from collections import Counter
from sklearn.decomposition import PCA
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import re
from matplotlib.patches import Patch
from PIL import Image
%matplotlib inline

# 42 is for TrueType fonts (editable text in Illustrator)
sc.settings.set_figure_params(dpi=300, dpi_save=300)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Define Working Directory
working_dic = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/immune_chimerism/results/"
os.chdir(working_dic)

sc.settings.verbosity = 1
sc.logging.print_header()

dimplot_color = mcolors.LinearSegmentedColormap.from_list("custom_cmap", ["#E5E5E5", "#4C4DB7"])
COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

COND_COLORS = {
    "Control": "#EFE48E",
    "3month-protocol": "skyblue",
    "6month-protocol": "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV": "#F0C096",
    "Rejection": "#E09E9E",
}

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import linregress, spearmanr
from sklearn.metrics import r2_score
from adjustText import adjust_text

In [2]:
adata = sc.read ('/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/global_object_assembly/objects/global_X_UMAP_Added_PCMV_score_updated_celltype_cellstate_annotated_preclinical_object_RAW_v3.h5ad')


In [3]:
adata

AnnData object with n_obs × n_vars = 233200 × 53826
    obs: 'Sample_ID', 'source', 'doublet_score', 'predicted_doublet_scrublet', 'n_counts_Sus_scrofa', 'n_counts_Papio_anubis', 'species_origin', 'barnyard_doublet', 'Survival', 'Condition', 'Donor_heart_mass (g)', 'Recipient_heart_mass (g)', 'Recipient_LVEF (%)', 'Recipient_LV_Wall_Thickness (mm)', 'TropT (ng/ml)', 'Immunosuppression_Protocol', 'Heart_Mass_Ratio', 'Exp', 'percent_mito', 'percent_ribo', 'n_genes_by_counts', 'total_counts', 'n_counts', 'n_genes', 'n_counts_PCMV', 'n_genes_PCMV', 'PCMV_score', 'cell_type', 'cell_state', 'n_counts_pCMV', 'n_genes_pCMV', 'pCMV_score', 'PCMV_detection_threshold'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'Condition_colors', 'Sample_ID_colors', 'cell_state_colors', 'cell_type_colors'
    obsm: 'X_pca_harmony', 'X_umap', 'X_umap_CM', 'X_umap_EC', 'X_umap_FB', 'X_umap_Lymph_papio', 'X_umap_Lymph_sus', 'X_

In [8]:
# Immune cell dynamics

RESULTS_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/immune_chimerism/results/immune_exchange"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.chdir(RESULTS_DIR)

# Visual identity
COLOR_DONOR     = "#F0A848"   
COLOR_RECIPIENT = "#1F4E79"  
FONT_MAIN       = 10
FONT_TICK        = 10
FONT_LABEL       = 7
SCATTER_SIZE     = 250
LINE_WIDTH       = 7
BAND_ALPHA       = 0.22    # opacity of the 95 % confidence band around each fit
N_BOOT           = 2000    # wild-bootstrap resamples for the confidence bands
# FIG_SIZE         = (5.6, 5.15)
FIG_SIZE         = (7.75, 6.75)
CONTROL_CONDITION = "Control"   # non-transplanted hearts: reference only, never fitted or tested

# Reproducibility
SEED = 42
np.random.seed(SEED)

obs_full = adata.obs.copy()
print("Data loaded successfully.")

# model deifnitions
def exp_decay(x, a, b, c):
    """Three-parameter exponential decay: y = a·exp(-b·x) + c"""
    return a * np.exp(-b * x) + c

def _fmt_p(p):
    return f"{p:.2f}" if p >= 0.01 else (f"{p:.3f}" if p >= 0.001 else f"{p:.1e}")


# Core function
def run_immune_dynamics_analysis(
    obs_full: pd.DataFrame,
    baboon_types: list,
    pig_types: list,
    label_suffix: str,
    title_suffix: str,
    results_dir: str,
) -> dict:
    """
    Build percentage data, fit models (xenografts only), compute statistics, plot, and export.
    """
    obs = obs_full.copy()
    obs["Survival"] = pd.to_numeric(obs["Survival"], errors="coerce")

    # assigning the immune groups
    conditions  = [
        obs["cell_type"].isin(baboon_types),
        obs["cell_type"].isin(pig_types),
    ]
    choices = ["Recipient_papio_immune", "Donor_sus_immune"]
    obs["immune_group"] = np.select(conditions, choices, default=None)
    obs.loc[obs["immune_group"] == "None", "immune_group"] = np.nan
    obs.loc[obs["immune_group"] == None,   "immune_group"] = np.nan

    # computing the percentages over all nuclei per sample
    total_per_sample = (
        obs.groupby("Sample_ID", observed=True).size().rename("total_nuc")
    )
    survival_per_sample = (
        obs.groupby("Sample_ID", observed=True)["Survival"].first()
    )
    condition_per_sample = (
        obs.groupby("Sample_ID", observed=True)["Condition"].first().astype(str)
    )

    group_counts = (
        obs[obs["immune_group"].notna()]
        .groupby(["Sample_ID", "immune_group"], observed=True)
        .size()
        .rename("n_cells")
        .reset_index()
    )
    group_counts = group_counts.merge(total_per_sample, on="Sample_ID", how="left")
    group_counts["percentage"] = group_counts["n_cells"] / group_counts["total_nuc"] * 100

    # Ensure every sample × group combination exists
    full_idx = pd.MultiIndex.from_product(
        [total_per_sample.index, ["Donor_sus_immune", "Recipient_papio_immune"]],
        names=["Sample_ID", "immune_group"],
    )
    group_counts = (
        group_counts.set_index(["Sample_ID", "immune_group"])
        .reindex(full_idx)
        .reset_index()
    )
    group_counts["percentage"]  = group_counts["percentage"].fillna(0)
    group_counts["n_cells"]     = group_counts["n_cells"].fillna(0).astype(int)
    group_counts["Survival"]    = group_counts["Sample_ID"].map(survival_per_sample)
    group_counts["total_nuc"]   = group_counts["Sample_ID"].map(total_per_sample)
    group_counts["Condition"]   = group_counts["Sample_ID"].map(condition_per_sample)
    group_counts["in_fit"]      = group_counts["Condition"] != CONTROL_CONDITION

    # Split: xenografts (fitted, tested) and Controls (reference only)
    def _part(group, fitted):
        d = group_counts[(group_counts["immune_group"] == group) & (group_counts["in_fit"] == fitted)]
        return d.dropna(subset=["Survival"]).copy()
    donor_df, recip_df = _part("Donor_sus_immune", True), _part("Recipient_papio_immune", True)
    donor_ctrl, recip_ctrl = _part("Donor_sus_immune", False), _part("Recipient_papio_immune", False)

    x_don = donor_df["Survival"].values.astype(float)
    y_don = donor_df["percentage"].values.astype(float)
    x_rec = recip_df["Survival"].values.astype(float)
    y_rec = recip_df["percentage"].values.astype(float)

    # Fit: donor (sus): exponential decay
    early_mask = x_don <= np.percentile(x_don, 25)
    p0 = [y_don[early_mask].mean() if early_mask.any() else y_don.max(), 0.1, 0.0]
    popt_don, _ = curve_fit(exp_decay, x_don, y_don, p0=p0,
                            bounds=([0, 1e-4, 0], [np.inf, 10.0, np.inf]), maxfev=50_000)
    y_don_pred = exp_decay(x_don, *popt_don)
    r2_don     = r2_score(y_don, y_don_pred)
    rho_don, p_don_spearman = spearmanr(x_don, y_don)

    # Fit: recipient (papio): linear
    lr = linregress(x_rec, y_rec)
    y_rec_pred  = lr.slope * x_rec + lr.intercept
    r2_rec      = r2_score(y_rec, y_rec_pred)
    rho_rec, p_rec_spearman = spearmanr(x_rec, y_rec)

    # 95 % confidence band of each fit (wild bootstrap over hearts)

    x_smooth_don = np.linspace(x_don.min(), x_don.max(), 300)
    x_smooth_rec = np.linspace(x_rec.min(), x_rec.max(), 300)
    rng = np.random.default_rng(SEED)
    res_don, res_rec = y_don - y_don_pred, y_rec - y_rec_pred
    boot_don, boot_rec, n_fail_don = [], [], 0
    for _ in range(N_BOOT):
        y_star = y_don_pred + res_don * rng.choice([-1.0, 1.0], len(x_don))
        try:
            pb, _ = curve_fit(exp_decay, x_don, y_star, p0=popt_don,
                              bounds=([0, 1e-4, 0], [np.inf, 10.0, np.inf]), maxfev=50_000)
            boot_don.append(exp_decay(x_smooth_don, *pb))
        except (RuntimeError, ValueError):
            n_fail_don += 1
        sl, ic = np.polyfit(x_rec, y_rec_pred + res_rec * rng.choice([-1.0, 1.0], len(x_rec)), 1)
        boot_rec.append(sl * x_smooth_rec + ic)
    don_lo, don_hi = np.percentile(np.vstack(boot_don), [2.5, 97.5], axis=0)
    rec_lo, rec_hi = np.percentile(np.vstack(boot_rec), [2.5, 97.5], axis=0)
    rec_lo = np.clip(rec_lo, 0, None)

    _all_d = group_counts[group_counts["immune_group"] == "Donor_sus_immune"].dropna(subset=["Survival"])
    _all_r = group_counts[group_counts["immune_group"] == "Recipient_papio_immune"].dropna(subset=["Survival"])
    rho_don_c, p_don_c = spearmanr(_all_d["Survival"], _all_d["percentage"])
    rho_rec_c, p_rec_c = spearmanr(_all_r["Survival"], _all_r["percentage"])

    #Summary dictionaries
    stats = {
        "compartment": label_suffix,
        "n_xenografts": len(x_don),
        "n_controls_reference_only": len(donor_ctrl),
        # Donor (sus): Spearman (the statistic) and exponential fit 
        "donor_Spearman_rho": rho_don,
        "donor_Spearman_pval": p_don_spearman,
        "donor_exp_a": popt_don[0],
        "donor_exp_b_decay_rate_per_day": popt_don[1],
        "donor_exp_half_life_days": np.log(2) / popt_don[1],
        "donor_exp_c_baseline": popt_don[2],
        "donor_exp_R2": r2_don,
        # Recipient (papio): Spearman (the statistic) and linear fit 
        "recipient_Spearman_rho": rho_rec,
        "recipient_Spearman_pval": p_rec_spearman,
        "recipient_slope_pct_per_day": lr.slope,
        "recipient_intercept": lr.intercept,
        "recipient_linear_R2": r2_rec,
        # 95 % confidence bands of the fits: wild bootstrap over hearts
        "CI_band_wild_bootstrap_resamples": N_BOOT,
        "donor_CI_band_failed_refits": n_fail_don,
        # traceability: the superseded numbers with the 4 Controls as day-0 points
        "donor_Spearman_rho_incl_controls": rho_don_c,
        "donor_Spearman_pval_incl_controls": p_don_c,
        "recipient_Spearman_rho_incl_controls": rho_rec_c,
        "recipient_Spearman_pval_incl_controls": p_rec_c,
    }

    # Plotting function

    def _make_plot(label_points: bool = False):
        fig, ax = plt.subplots(figsize=FIG_SIZE)
        fig.patch.set_facecolor("none")
        ax.set_facecolor("none")

        # Scatter: xenografts (filled)
        ax.scatter(
            x_don, y_don,
            facecolor=COLOR_DONOR, edgecolor="black", linewidth=0.6,
            s=SCATTER_SIZE, alpha=0.92, zorder=3,
            label="Donor (sus) immune cells",
        )
        ax.scatter(
            x_rec, y_rec,
            facecolor=COLOR_RECIPIENT, edgecolor="black", linewidth=0.6,
            s=SCATTER_SIZE, alpha=0.92, marker="s", zorder=3,
            label="Recipient (papio) immune cells",
        )
        # Controls: open markers at day 0, reference only (not fitted, not tested)
        ax.scatter(
            donor_ctrl["Survival"], donor_ctrl["percentage"],
            facecolor="none", edgecolor=COLOR_DONOR, linewidth=1.4, s=SCATTER_SIZE, zorder=4,
        )
        ax.scatter(
            recip_ctrl["Survival"], recip_ctrl["percentage"],
            facecolor="none", edgecolor=COLOR_RECIPIENT, linewidth=1.4, s=SCATTER_SIZE, marker="s", zorder=4,
            label=f"Control hearts (open; reference, not fitted, n = {len(donor_ctrl)})",
        )

        # 95 % confidence bands of the fits, each in the colour of its own line
        ax.fill_between(x_smooth_don, don_lo, don_hi, color=COLOR_DONOR, alpha=BAND_ALPHA, lw=0, zorder=1)
        ax.fill_between(x_smooth_rec, rec_lo, rec_hi, color=COLOR_RECIPIENT, alpha=BAND_ALPHA, lw=0, zorder=1)

        # Fit curves with the Spearman statistic of the xenografts
        ax.plot(
            x_smooth_don,
            exp_decay(x_smooth_don, *popt_don),
            color=COLOR_DONOR, linewidth=LINE_WIDTH, zorder=2, solid_capstyle="round",
            label=(
                f"Donor exp. fit R²={r2_don:.2f}; "
                f"Spearman ρ={rho_don:.2f}, P={_fmt_p(p_don_spearman)}"
            ),
        )
        ax.plot(
            x_smooth_rec,
            lr.slope * x_smooth_rec + lr.intercept,
            color=COLOR_RECIPIENT, linewidth=LINE_WIDTH, zorder=2, solid_capstyle="round",
            label=(
                f"Recipient linear fit R²={r2_rec:.2f}; "
                f"Spearman ρ={rho_rec:.2f}, P={_fmt_p(p_rec_spearman)}"
            ),
        )

        # Axes
        ax.set_xlabel("Survival (days)", fontsize=FONT_MAIN, fontweight="bold")
        ax.set_ylabel("% of all nuclei", fontsize=FONT_MAIN, fontweight="bold")
        ax.set_title(
            f"{title_suffix} Dynamics vs. Survival",
            fontsize=FONT_MAIN, fontweight="bold", pad=14,
        )
        for spine in ["top", "bottom", "left", "right"]:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_linewidth(1.75)
            ax.spines[spine].set_color("black")

        ax.tick_params(labelsize=FONT_TICK)
        ax.grid(False)

        # Legends, statistics are for the xenografts (n stated)
        handles, labels = ax.get_legend_handles_labels()
        handles.append(Patch(facecolor="0.55", alpha=BAND_ALPHA + 0.15, lw=0))
        labels.append(f"shaded: 95 % CI of each fit (wild bootstrap over hearts, {N_BOOT:,} resamples)")
        leg = ax.legend(
            handles, labels, markerscale=0.45,
            loc="upper left" if lr.slope >= 0 else "best",
            fontsize=7.5,
            frameon=True,
            title=f"fits and Spearman: xenografts only (n = {len(x_don)})",
            title_fontsize=7.5,
        )
        leg._legend_box.align = "left"

        # Sample labels
        if label_points:
            texts = []
            for df_, col in [(donor_df, COLOR_DONOR), (donor_ctrl, COLOR_DONOR), (recip_df, COLOR_RECIPIENT), (recip_ctrl, COLOR_RECIPIENT)]:
                for _, row in df_.iterrows():
                    texts.append(
                        ax.text(
                            row["Survival"], row["percentage"],
                            str(row["Sample_ID"]),
                            fontsize=FONT_LABEL, color=col, fontweight="bold",
                        )
                    )
            adjust_text(
                texts, ax=ax,
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.4, alpha=0.6),
                expand=(1.2, 1.4),
                force_text=(0.3, 0.5),
            )

        fig.tight_layout()
        return fig

    # Save clean plot
    fig_clean = _make_plot(label_points=False)
    fig_clean.savefig(
        os.path.join(results_dir, f"Immune_dynamics_{label_suffix}.pdf"),
        dpi=300, bbox_inches="tight", transparent=True,
    )
    plt.close(fig_clean)

    # Save labeled plot
    fig_label = _make_plot(label_points=True)
    fig_label.savefig(
        os.path.join(results_dir, f"Immune_dynamics_{label_suffix}_labeled.pdf"),
        dpi=300, bbox_inches="tight", transparent=True,
    )
    plt.close(fig_label)

    # Per-sample CSVs
    export = group_counts.copy()
    export["fit_type"] = np.where(
        export["immune_group"] == "Donor_sus_immune", "Exponential", "Linear"
    )
    export["fitted_pct"] = export.apply(
        lambda r: (
            np.nan if not r["in_fit"]
            else exp_decay(r["Survival"], *popt_don)
            if r["immune_group"] == "Donor_sus_immune"
            else lr.slope * r["Survival"] + lr.intercept
        ),
        axis=1,
    )
    export["residual"] = export["percentage"] - export["fitted_pct"]
    export = export.sort_values(["immune_group", "Survival", "Sample_ID"]).reset_index(
        drop=True
    )
    export.to_csv(
        os.path.join(results_dir, f"Immune_dynamics_{label_suffix}_per_sample.csv"),
        index=False,
    )

    print(f"\n{'─'*60}")
    print(f"  {label_suffix}  |  {title_suffix}  |  xenografts n = {len(x_don)} (Controls n = {len(donor_ctrl)} shown, not fitted)")
    print(f"{'─'*60}")
    print(f"  Donor (sus):      Spearman ρ = {rho_don:.3f},  P = {p_don_spearman:.2e}   "
          f"[exp. fit a = {popt_don[0]:.3f}, b = {popt_don[1]:.4f}/day (half-life {np.log(2)/popt_don[1]:.1f} d), c = {popt_don[2]:.3f}, R² = {r2_don:.3f}]")
    print(f"  Recipient (papio): Spearman ρ = {rho_rec:.3f},  P = {p_rec_spearman:.2e}   "
          f"[linear slope = {lr.slope:.4f} %/day, R² = {r2_rec:.3f}]")
    print(f"  superseded (Controls as day-0 points): donor ρ = {rho_don_c:.3f} (P = {p_don_c:.1e}), recipient ρ = {rho_rec_c:.3f} (P = {p_rec_c:.1e})")
    print(f"{'─'*60}")

    return stats

# run all here

all_stats = []

# 1) All immune cells
all_stats.append(
    run_immune_dynamics_analysis(
        obs_full,
        baboon_types=["Mye_papio", "Lymph_papio"],
        pig_types=["Mye_sus", "Lymph_sus"],
        label_suffix="ALL",
        title_suffix="All Immune Cells",
        results_dir=RESULTS_DIR,
    )
)

# 2) Myeloid only
all_stats.append(
    run_immune_dynamics_analysis(
        obs_full,
        baboon_types=["Mye_papio"],
        pig_types=["Mye_sus"],
        label_suffix="Myeloid",
        title_suffix="Myeloid Compartment",
        results_dir=RESULTS_DIR,
    )
)

# 3) Lymphoid only
all_stats.append(
    run_immune_dynamics_analysis(
        obs_full,
        baboon_types=["Lymph_papio"],
        pig_types=["Lymph_sus"],
        label_suffix="Lymphoid",
        title_suffix="Lymphoid Compartment",
        results_dir=RESULTS_DIR,
    )
)

# export the CSVs w statistics
stats_df = pd.DataFrame(all_stats)
stats_path = os.path.join(RESULTS_DIR, "Immune_dynamics_fit_statistics.csv")
stats_df.to_csv(stats_path, index=False)
print(f"\n✓  Combined statistics saved → {stats_path}")
print(stats_df.round(4).to_string(index=False))


Data loaded successfully.
50 [-0.25091976  0.90142861]
51 [0.46398788 0.19731697]
48 [-0.68796272 -0.68801096]
49 [-0.88383278  0.73235229]
1 [0.20223002 0.41614516]
11 [-0.95883101  0.9398197 ]

────────────────────────────────────────────────────────────
  ALL  |  All Immune Cells  |  xenografts n = 22 (Controls n = 4 shown, not fitted)
────────────────────────────────────────────────────────────
  Donor (sus):      Spearman ρ = -0.730,  P = 1.14e-04   [exp. fit a = 6.058, b = 0.0697/day (half-life 9.9 d), c = 0.033, R² = 0.852]
  Recipient (papio): Spearman ρ = 0.756,  P = 4.63e-05   [linear slope = 0.0719 %/day, R² = 0.574]
  superseded (Controls as day-0 points): donor ρ = -0.763 (P = 5.8e-06), recipient ρ = 0.845 (P = 5.6e-08)
────────────────────────────────────────────────────────────
50 [ 0.66488528 -0.57532178]
51 [-0.63635007 -0.63319098]
48 [-0.39151551  0.04951286]
49 [-0.13610996 -0.41754172]
1 [ 0.22370579 -0.72101228]
11 [-0.4157107  -0.26727631]
6 [-0.08786003  0.57035

In [6]:
mapping_df = adata.obs[['Sample_ID', 'Survival', 'Condition']].copy()

# Drop duplicates to only keep unique Sample_ID entries
unique_mapping = mapping_df.drop_duplicates().reset_index(drop=True)

# Show the table
print(unique_mapping)


   Sample_ID  Survival               Condition
0     5411-2         0                 Control
1     5806-2         0                 Control
2       7159         0                 Control
3       7518         0                 Control
4      PAV65        90         3month-protocol
5      PAV66        51         3month-protocol
6      PAV67        90         3month-protocol
7      PAV68        11                PCMV/PRV
8      PAV69        90         3month-protocol
9      PAV71       182         6month-protocol
10     PAV72       195         6month-protocol
11     PAV73        15                PCMV/PRV
12     PAV74        27                PCMV/PRV
13     PAV75        90         3month-protocol
14     PAV76        18               Rejection
15     PAV78        51               Rejection
16     PAV79        29               Rejection
17     PAV80        90         3month-protocol
18     PAV82        90         3month-protocol
19     PAV83         1  Prematurely-terminated
20     PAV84 